# SparseWalker — streaming-native semantics

This experiment fixes the training/serving mismatch found in the latency audit.

**Model semantics:** persistent local Walker state + two cacheable temporal layers, no learned absolute positions. Temporal ordering is represented by a query-time relative-lag bias, so cached K/V never need re-encoding when the history advances.

Training uses full chronological user histories in 64-event chunks. Forward state is carried exactly across chunks; gradients are detached at chunk boundaries (TBPTT), and parameters are updated only after the full user batch has streamed through.

This first run uses exact dense reads over a 512-event temporal memory. If quality survives, the next step is replacing that read with the already validated SWG Top-16 search and benchmarking the truly quality-equivalent persistent serving path.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'; BRANCH='agent/walker-streaming-native'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src'); sys.path.insert(0,f'{REPO}/experiments')
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


In [ ]:
import runpy
SCRIPT=f'{REPO}/experiments/run_ml1m_walker_streaming_native.py'
sys.argv=[SCRIPT,
          '--epochs','15',
          '--eval-every','5',
          '--batch-size','64',
          '--eval-batch-size','64',
          '--chunk-size','64',
          '--memory-size','512']
print('STREAMING WALKER START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('STREAMING WALKER END',flush=True)


The run must pass `CHUNK_INVARIANCE` and `CAUSAL_TEST` before training. Paste the first `TRAIN`/`EVAL` blocks as they appear. The key metric is `val_stream_full_NDCG@10`; `persistent_history_gain` compares full persistent history against resetting the same model to only the last 200 events.


In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_streaming_native/ml1m/seed42/result.json')
if p.exists():
    r=json.loads(p.read_text())
    print('FINAL',json.dumps(r,indent=2))
else:
    print('No final result yet; inspect TRAIN/EVAL output above.')
